In [ ]:
# Enterprise FinTech Payment Intelligence Platform
## Phase 4 – Machine Learning
### Notebook 01 – Data Preparation

**Objective:**
Extract data from the SQL Server Data Warehouse via the Semantic View layer, prepare a clean analytical dataset, and export it for Machine Learning modeling.

In [1]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Enterprise_FinTech_Payment_Intelligence;"
    "Trusted_Connection=yes;"
)
print("Connected Successfully")


Connected Successfully


In [3]:
# Enterprise Best Practice: Querying the pre-aggregated SQL View
query = "SELECT * FROM vw_PaymentSummary"

print("Extracting data... this may take a moment for 6.3M rows.")
df = pd.read_sql(query, conn)

# Display the first 5 rows
df.head()

Extracting data... this may take a moment for 6.3M rows.


,TransactionID,DayNumber,HourOfSimulation,PeriodOfDay,TransactionType,SourceAccountID,DestinationAccountID,IsFraud,IsFlaggedFraud,Amount,OldBalanceOrig,NewBalanceOrig,OldBalanceDest,NewBalanceDest
0,20666,1,13,Afternoon,PAYMENT,C2108058134,M1963634359,False,False,22005.03,0.00,0.00,0.00,0.00
1,20851,1,15,Afternoon,CASH_IN,C1672847392,C882180306,False,False,214524.46,5030.00,219554.46,0.00,0.00
2,21169,1,15,Afternoon,TRANSFER,C519692057,C834458122,False,False,1432648.47,0.00,0.00,1453236.68,2885885.15
3,21181,1,15,Afternoon,CASH_OUT,C1634465843,C807329566,False,False,352807.10,0.00,0.00,426401.78,779208.88
4,21321,1,15,Afternoon,CASH_OUT,C210125456,C410910993,False,False,275348.64,390959.31,115610.67,0.00,275348.64


In [4]:
# Verify dataset dimensions
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 6,362,620
Columns: 14


In [5]:
# Check memory usage and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 14 columns):
 #   Column                Dtype  
---  ------                -----  
 0   TransactionID         int64  
 1   DayNumber             int64  
 2   HourOfSimulation      int64  
 3   PeriodOfDay           object 
 4   TransactionType       object 
 5   SourceAccountID       object 
 6   DestinationAccountID  object 
 7   IsFraud               bool   
 8   IsFlaggedFraud        bool   
 9   Amount                float64
 10  OldBalanceOrig        float64
 11  NewBalanceOrig        float64
 12  OldBalanceDest        float64
 13  NewBalanceDest        float64
dtypes: bool(2), float64(5), int64(3), object(4)
memory usage: 594.7+ MB


In [6]:
# Verify data cleanliness
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


TransactionID           0
DayNumber               0
HourOfSimulation        0
PeriodOfDay             0
TransactionType         0
SourceAccountID         0
DestinationAccountID    0
IsFraud                 0
IsFlaggedFraud          0
Amount                  0
OldBalanceOrig          0
NewBalanceOrig          0
OldBalanceDest          0
NewBalanceDest          0
dtype: int64

In [7]:
# Ensure numerical and categorical types are properly aligned
df.dtypes

TransactionID             int64
DayNumber                 int64
HourOfSimulation          int64
PeriodOfDay              object
TransactionType          object
SourceAccountID          object
DestinationAccountID     object
IsFraud                    bool
IsFlaggedFraud             bool
Amount                  float64
OldBalanceOrig          float64
NewBalanceOrig          float64
OldBalanceDest          float64
NewBalanceDest          float64
dtype: object

In [8]:
# Quick statistical overview of numerical columns
df.describe().apply(lambda s: s.apply('{0:.2f}'.format))

,TransactionID,DayNumber,HourOfSimulation,Amount,OldBalanceOrig,NewBalanceOrig,OldBalanceDest,NewBalanceDest
count,6362620.00,6362620.00,6362620.00,6362620.00,6362620.00,6362620.00,6362620.00,6362620.00
mean,3181310.50,10.49,15.59,179861.90,833883.10,855113.67,1100701.67,1224996.40
std,1836730.33,5.92,4.10,603858.23,2888242.67,2924048.50,3399180.11,3674128.94
min,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00
25%,1590655.75,7.00,12.00,13389.57,0.00,0.00,0.00,0.00
50%,3181310.50,10.00,16.00,74871.94,14208.00,0.00,132705.66,214661.44
75%,4771965.25,14.00,19.00,208721.48,107315.18,144258.41,943036.71,1111909.25
max,6362620.00,31.00,24.00,92445516.64,59585040.37,49585040.37,356015889.35,356179278.92


In [9]:
# Check the class imbalance (Target Variable)
fraud_counts = df["IsFraud"].value_counts()
fraud_percentages = df["IsFraud"].value_counts(normalize=True) * 100

distribution_df = pd.DataFrame({
    'Transaction Count': fraud_counts,
    'Percentage (%)': fraud_percentages.round(3)
})

print("Target Variable Distribution (IsFraud):")
display(distribution_df)

Target Variable Distribution (IsFraud):


,Transaction Count,Percentage (%)
IsFraud,,
False,6354407,99.871
True,8213,0.129


In [10]:
# Save the clean analytical dataset for the next step
output_filename = "clean_ml_dataset.csv"

print(f"Exporting dataset to {output_filename}...")
df.to_csv(output_filename, index=False)
print("Dataset Exported Successfully")

Exporting dataset to clean_ml_dataset.csv...
Dataset Exported Successfully


In [11]:
print("=" * 60)
print("Data Preparation Completed Successfully")
print("=" * 60)

print(f"Rows Loaded        : {df.shape[0]:,}")
print(f"Columns Loaded     : {df.shape[1]}")
print(f"Fraud Transactions : {df['IsFraud'].sum():,}")
print(f"Output File        : clean_ml_dataset.csv")

print("=" * 60)

Data Preparation Completed Successfully
Rows Loaded        : 6,362,620
Columns Loaded     : 14
Fraud Transactions : 8,213
Output File        : clean_ml_dataset.csv


In [12]:
try:
    conn.close()
    print("SQL Server Connection Closed")
except:
    print("Connection was already closed")


SQL Server Connection Closed
